# Homosaurus Vocab Markdown Browswer

This is a Python notebook that converts the [Homosaurus v5 vocabulary](https://homosaurus.org/v5) into markdown files for browsing in an Obsidian vault.

See the README.md file for instructions on what the script should do.

In [ ]:
from pathlib import Path
import re

In [ ]:
def strip_quotes(s):
    if not s:
        return ""
    
    s = s.replace('"""', '"')
    s = s.replace('\\"', "'")
    m = re.search(r'"(.*?)"', s)
    if m:
        return m.group(1).strip()
    return s.strip().strip('.,;')

def extract_lang_value(val, lang_tag='@en'):
    if not val:
        return ""
    candidates = val if isinstance(val, list) else [val]
    for item in candidates:
        if lang_tag in item:
            return strip_quotes(item)
    # Return no value if expected lang_tag not found
    return ""

def make_filename(term):
    pref = extract_lang_value(term.get('skos:prefLabel'))
    filename = pref or strip_quotes(term.get('dc:identifier', '')) or term.get('uri', '')
    # sanitize filename
    filename = re.sub(r'[\\/:"*?<>|]+', '', filename).strip()
    return filename

In [ ]:
# Prior to importing the file it was necessary to do some minor data clean up directly in the file, specifically:
# 1) conjoining certain lines that had errant linebreaks that broke up their contents into two lines, and
# 2) removing multiple nested levels of both escaped and unescaped quotes around an element value.
#
# The data file in the data/ folder is the cleaned pre-processed file. Other data quality cleanup
# occurs in-line in the script.

# Read the data/v5.ttl file into memory and parse into a list of dicts.
# The first 6 lines (@prefix lines and whitespace) are discarded.
# Each object starts with a whitespace line and then its first line ends with `a skos:Concept;`
with open('data/v5.ttl', 'r') as file:
    lines = file.readlines()

# Skip the first 6 lines (@prefix declarations and whitespace)
lines = lines[6:]

# Parse the TTL file into a list of dicts
terms = []
current_term = {}

for line in lines:
    line = line.strip()
    
    if not line:
        # Empty line indicates start of new concept
        if current_term:
            terms.append(current_term)
            current_term = {}
    elif 'a skos:Concept' in line:
        # Extract the concept URI
        uri = line.split()[0].rstrip(';')
        current_term['uri'] = uri
    else:
        # Parse property and object
        if ':' in line and not line.startswith('@') and not line.startswith('<'):
            parts = line.split(None, 1)
            if len(parts) == 2:
                property = parts[0]
                value = parts[1].rstrip(';')
                current_term[property] = value
        else:
            # Handle case where line is a continuation of the previous property
            # by turning the property into a list and appending the trailing lines
            if property in current_term:
                if isinstance(current_term[property], list):
                    current_term[property].append(line)
                else:
                    current_term[property] = [current_term[property], line]
            else:
                current_term[property] = [line]

# Add the last term if exists
if current_term:
    terms.append(current_term)

print(f"Parsed {len(terms)} terms")

import json
with open('data.json', 'w') as f:
    json.dump(terms, f)

In [ ]:
# Create a map from URIs to filenames so we can look them up later
filename_map = {}
for term in terms:
    filename = make_filename(term);
    uri = term.get("uri")
    filename_map[uri] = filename

# Add missing values.
#
# It looks like these identifiers were redirected to terms with different identifiers on the
# web-hosted homosaurus vocab, but the ttl file was never corrected to use the canonical ids.
#
# <https://homosaurus.org/v5/homoit0000907> --> "LGBTQ+ museums" (actually homoit0001030)
# <https://homosaurus.org/v5/homoit0000337> --> "LGBTQ+ Deaf people" (actually homoit0001866)
filename_map["<https://homosaurus.org/v5/homoit0000907>"] = "LGBTQ+ museums";
filename_map["<https://homosaurus.org/v5/homoit0000337>"] = "LGBTQ+ Deaf people";

# uri_filename_map

In [ ]:
# Write the list of dicts as individual markdown files in the `vault` folder.

# Each term entry in the list is a new markdown file, and the file should be named by the prefLabel property
# from the list of prefLabels, but the one that ends with @en, and only include the value inside the quotes as the file name.

# The markdown file starts with yaml frontmatter as per Obsidian frontmatter formatting.
# The frontmatter includes the following properties:
# 
# id (pulled from dc:identifier)
# URL (pulled from url)
# prefLabel fr (this is an empty frontmatter property)
# status (this is an empty frontmatter property)
# category (this is an empty frontmatter property)

# The markdown body contains the following contents:
#
# ## Description (en)
# (pulled from rdfs:comment property that ends with @en, only including the value inside the quotes)
#
# ## Description (fr)
# (leave this empty)
#
# ## altLabel (en)
# (pulled from skos:altLabel property that ends with @en, if it exists, only the value inside the quotes)
#
# ## altLabel (fr)
# (leave this empty)
# ## Relationships
#
# ### Broader
# (pulled from skos:broader)
# 
# ### Narrower
# (pulled from skos:narrower)
#
# ### Related
# (pulled from skos:related)
#
# ## Comments
# (leave this empty)

def convert_filename_wikilinks(val):
    if not val:
        return []
    candidates = val if isinstance(val, list) else [val]
    filenames = []
    for item in candidates:
        index = item.index('>')
        item = item[:index + 1]
        filename = filename_map.get(item, None)
        filenames.append(f"[[{filename}]]")
        if not filename:
            print(f"Missing item: {item}")
    return filenames
    
vault_dir = Path("vault")
vault_dir.mkdir(exist_ok=True)

for term in terms:
    # filename from skos:prefLabel @en
    filename = make_filename(term)
    filepath = vault_dir / f"{filename}.md"

    identifier = strip_quotes(term.get('dc:identifier', ''))
    url_val = term.get('url') or term.get('uri') or ""
    url = url_val.strip('<>')
    
    desc_en = extract_lang_value(term.get('rdfs:comment'), '@en')
    alt_en = extract_lang_value(term.get('skos:altLabel'), '@en')
    pref_label_fr=extract_lang_value(term.get('skos:prefLabel'), '@fr')

    broader = convert_filename_wikilinks(term.get('skos:broader'))
    narrower = convert_filename_wikilinks(term.get('skos:narrower'))
    related = convert_filename_wikilinks(term.get('skos:related'))

    lines = []
    lines.append('---')
    lines.append(f"id: {identifier}")
    lines.append(f"URL: {url}")
    lines.append(f"prefLabel fr: {pref_label_fr}")
    lines.append("status: ")
    lines.append("category: ")
    lines.append('---\n')

    
    lines.append("## Description (en)\n")
    lines.append(desc_en or "(n/a)")

    lines.append("\n## Description (fr)\n\n==FILL IN==\n")
    
    lines.append("## altLabel (en)\n")
    lines.append(alt_en or "(n/a)")

    lines.append("\n## altLabel (fr)\n\n==FILL IN==\n")

    lines.append("## Relationships")
    

    if broader:
        lines.append("\n### Broader\n")
        for u in broader:
            lines.append(f"- {u}")
    if narrower:
        lines.append("\n### Narrower\n")
        for u in narrower:
            lines.append(f"- {u}")
    if related:
        lines.append("\n### Related\n")
        for u in related:
            lines.append(f"- {u}")

    lines.append("\n## Comments\n")

    with open(filepath, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
